In [1]:
# repo rate cleaner

import pandas as pd
from datetime import timedelta
import calendar

def process_repo_rate_3m_change(output_path, start_year=2010, end_year=2026):

    # 1. Repo rate timeline — same as existing T-bill cleaner
    repo_timeline = {
        '2025-12-05': 5.25, '2025-06-06': 5.50, '2025-04-09': 6.00, '2025-02-07': 6.25,
        '2023-02-08': 6.50, '2022-12-07': 6.25, '2022-09-30': 5.90, '2022-08-05': 5.40,
        '2022-06-08': 4.90, '2022-05-04': 4.40, '2020-05-22': 4.00, '2020-03-27': 4.40,
        '2019-10-04': 5.15, '2019-08-07': 5.40, '2019-06-06': 5.75, '2019-04-04': 6.00,
        '2019-02-07': 6.25, '2018-08-01': 6.50, '2018-06-06': 6.25, '2017-08-02': 6.00,
        '2016-10-04': 6.25, '2016-04-05': 6.50, '2015-09-29': 6.75, '2015-06-02': 7.25,
        '2015-03-04': 7.50, '2015-01-15': 7.75, '2014-01-28': 8.00, '2013-10-29': 7.75,
        '2013-09-20': 7.50, '2013-05-03': 7.25, '2013-03-19': 7.50, '2013-01-29': 7.75,
        '2012-04-17': 8.00, '2011-10-25': 8.50, '2011-09-16': 8.25, '2011-07-26': 8.00,
        '2011-06-16': 7.50, '2011-05-03': 7.25, '2011-03-17': 6.75, '2011-01-25': 6.50,
        '2010-11-02': 6.25, '2010-09-16': 6.00, '2010-07-27': 5.75, '2010-07-02': 5.50,
        '2010-04-20': 5.25, '2010-03-19': 5.00, '2009-04-21': 4.75, '2009-03-04': 5.00,
        '2009-01-02': 5.50, '2008-12-06': 6.50, '2008-11-01': 7.50, '2008-10-20': 8.00,
        '2008-07-29': 9.00, '2008-06-24': 8.75, '2008-06-11': 8.00, '2007-03-30': 7.75,
        '2007-01-31': 7.50, '2006-10-31': 7.25, '2006-07-25': 7.00, '2006-06-09': 6.75,
        '2006-01-24': 6.50, '2005-10-25': 6.25
    }

    repo_df = pd.DataFrame(list(repo_timeline.items()), columns=['Effective_Date', 'Repo_Rate'])
    repo_df['Effective_Date'] = pd.to_datetime(repo_df['Effective_Date'])
    repo_df = repo_df.sort_values('Effective_Date').reset_index(drop=True)

    # 2. Build monthly date spine — last day of each month
    months = pd.date_range(
        start=f'{start_year}-01-01',
        end=f'{end_year}-12-31',
        freq='MS'
    )
    last_days = [pd.Timestamp(d.year, d.month, calendar.monthrange(d.year, d.month)[1]) for d in months]
    monthly_df = pd.DataFrame({'Last_Day_of_Month': last_days})

    # 3. Point-in-time repo rate lookup for each month end
    monthly_df = monthly_df.sort_values('Last_Day_of_Month')
    monthly_df = pd.merge_asof(
        monthly_df,
        repo_df,
        left_on='Last_Day_of_Month',
        right_on='Effective_Date',
        direction='backward'
    )
    monthly_df['Repo_Rate'] = monthly_df['Repo_Rate'].fillna(5.00)

    # 4. Compute 3M change — current month repo rate minus repo rate 3 months prior
    monthly_df['Repo_Rate_3M_Change'] = monthly_df['Repo_Rate'] - monthly_df['Repo_Rate'].shift(3)
    monthly_df['Repo_Rate_3M_Average'] = monthly_df['Repo_Rate_3M_Change'].rolling(window=3).mean()

    # 5. Release date = same as period date (repo rate is known and effective immediately)
    monthly_df['Release_Date'] = monthly_df['Last_Day_of_Month']

    # 6. Format and save
    monthly_df['Last_Day_of_Month'] = monthly_df['Last_Day_of_Month'].dt.strftime('%Y-%m-%d')
    monthly_df['Release_Date'] = monthly_df['Release_Date'].dt.strftime('%Y-%m-%d')

    output_cols = ['Last_Day_of_Month', 'Release_Date', 'Repo_Rate', 'Repo_Rate_3M_Average']
    final_df = monthly_df[output_cols].dropna(subset=['Repo_Rate_3M_Average'])
    final_df = final_df.sort_values('Last_Day_of_Month', ascending=False)

    final_df.to_csv(output_path, index=False)
    print(f"Repo rate 3M change file saved to: {output_path}")
    print(f"Rows: {len(final_df)}")
    print(f"\nSample — key periods:")
    check_dates = ['2022-05-31', '2020-05-31', '2019-10-31', '2013-09-30', '2010-12-31']
    for d in check_dates:
        row = final_df[final_df['Last_Day_of_Month'] == d]
        if not row.empty:
            print(f"  {d}: Repo={row['Repo_Rate'].values[0]:.2f} | 3M_Change={row['Repo_Rate_3M_Average'].values[0]:.2f}")

from datetime import date
process_repo_rate_3m_change('Repo_Rate_3M_Average.csv', end_year=date.today().year)

Repo rate 3M change file saved to: Repo_Rate_3M_Average.csv
Rows: 199

Sample — key periods:
  2022-05-31: Repo=4.40 | 3M_Change=0.13
  2020-05-31: Repo=4.00 | 3M_Change=-0.88
  2019-10-31: Repo=5.15 | 3M_Change=-0.52
  2013-09-30: Repo=7.50 | 3M_Change=0.00
  2010-12-31: Repo=6.25 | 3M_Change=0.33


In [2]:
import pandas as pd
from pandas.tseries.offsets import BMonthEnd

# 1. Load the CSV file (skipping initial metadata rows)
input_file = "Month-end Yield of SGL Transactions in Government Dated Securities for Various Maturities.csv"
output_file = "Processed_G-Sec_Yield_With_Metrics.csv"

df = pd.read_csv(input_file, skiprows=6)
df.rename(columns={df.columns[0]: 'Year_Month'}, inplace=True)

# 2. Parse dates and clean footer rows
df['Datetime'] = pd.to_datetime(df['Year_Month'], format='%b-%Y', errors='coerce')
df.dropna(subset=['Datetime'], inplace=True)

# 3. CRITICAL: Sort chronologically to calculate the 3-month rolling difference accurately
df.sort_values('Datetime', ascending=True, inplace=True)

# Ensure yield columns are treated as numbers
for col in ['3 Years', '5 Years', '10 Years']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Calculate your new metrics
# .diff(3) calculates the change over 3 months since rows are sequential
df['10Y_3M_Chg'] = df['10 Years'].diff(3)
# Slope calculation: 10-year minus 3-year yield
df['10Y_3Y_Slope'] = df['10 Years'] - df['3 Years']

# 5. Filter for the requested timeframe: Jan 2010 to May 2026
df = df[(df['Datetime'] >= '2010-01-01') & (df['Datetime'] <= '2026-05-31')]

# 6. Generate the adjusted period and release dates
df['period_date'] = df['Datetime'] + BMonthEnd(0)
df['release_date'] = df['period_date']

# Format dates to YYYY-MM-DD strings
df['period_date'] = df['period_date'].dt.strftime('%Y-%m-%d')
df['release_date'] = df['release_date'].dt.strftime('%Y-%m-%d')

# 7. Optional: Sort back to reverse-chronological order (newest dates on top)
df.sort_values('Datetime', ascending=False, inplace=True)

# 8. Select and rename the final columns
final_columns = ['period_date', 'release_date', '3 Years', '5 Years', '10 Years', '10Y_3M_Chg', '10Y_3Y_Slope']
df_final = df[final_columns].copy()
df_final.columns = ['period_date', 'release_date', '3Y', '5Y', '10Y', '10Y_3M_Chg', '10Y_3Y_Slope']

# 9. Save to a new CSV file
df_final.to_csv(output_file, index=False)

print(f"File successfully created and saved as: '{output_file}'")
print(df_final.head(10))

File successfully created and saved as: 'Processed_G-Sec_Yield_With_Metrics.csv'
  period_date release_date      3Y      5Y     10Y  10Y_3M_Chg  10Y_3Y_Slope
0  2026-05-29   2026-05-29  6.5990  6.8027  7.0267      0.3148        0.4277
1  2026-04-30   2026-04-30  6.4202  6.8026  7.0796      0.2999        0.6594
2  2026-03-31   2026-03-31  6.4843  6.8393  7.0782      0.4192        0.5939
3  2026-02-27   2026-02-27  5.9129  6.3324  6.7119      0.1091        0.7990
4  2026-01-30   2026-01-30  6.0262  6.4688  6.7797      0.1545        0.7535
5  2025-12-31   2025-12-31  5.8866  6.3943  6.6590      0.0188        0.7724
6  2025-11-28   2025-11-28  5.8215  6.2557  6.6028     -0.0419        0.7813
7  2025-10-31   2025-10-31  5.9195  6.2456  6.6252      0.2020        0.7057
8  2025-09-30   2025-09-30  5.8634  6.2447  6.6402      0.2715        0.7768
9  2025-08-29   2025-08-29  6.0339  6.3770  6.6447      0.4246        0.6108


In [3]:
import pandas as pd
from pandas.tseries.offsets import BMonthEnd, BDay

# 1. Load and Clean CPI Data
cpi_df = pd.read_csv("cpi_220 (1).csv")
# Clean hidden whitespaces in month names just in case
cpi_df['month'] = cpi_df['month'].astype(str).str.strip()
cpi_df = cpi_df.dropna(subset=['year', 'month', 'inflation'])

# Keep the top-level inflation rate by dropping any subgroup duplicates for the same month
cpi_df = cpi_df[['year', 'month', 'inflation']].drop_duplicates(subset=['year', 'month'])

# Convert year + month to a Datetime object
cpi_df['Datetime'] = pd.to_datetime(cpi_df['year'].astype(str) + '-' + cpi_df['month'], format='%Y-%B', errors='coerce')
cpi_df.dropna(subset=['Datetime'], inplace=True)
cpi_df = cpi_df[(cpi_df['Datetime'] >= '2010-01-01') & (cpi_df['Datetime'] <= '2026-05-31')]

# 2. Date Operations for CPI
# End of the month business day
cpi_df['period_date'] = cpi_df['Datetime'] + BMonthEnd(0)

# Release date: 14th of the following month
cpi_df['raw_release'] = cpi_df['Datetime'] + pd.DateOffset(months=1, day=14)

# Roll the release date forward to the nearest business day if it lands on a weekend
cpi_df['cpi_release_date'] = cpi_df['raw_release'].apply(lambda x: BDay().rollforward(x))

# 3. Load and Clean G-Sec Yield Data
gsec_df = pd.read_csv("Month-end Yield of SGL Transactions in Government Dated Securities for Various Maturities.csv", skiprows=6)
gsec_df.rename(columns={gsec_df.columns[0]: 'Year_Month'}, inplace=True)
gsec_df['Datetime'] = pd.to_datetime(gsec_df['Year_Month'], format='%b-%Y', errors='coerce')
gsec_df.dropna(subset=['Datetime'], inplace=True)
gsec_df['10Y_Yield'] = pd.to_numeric(gsec_df['10 Years'], errors='coerce')

# 4. Merge CPI and G-Sec Yields
merged = pd.merge(cpi_df, gsec_df[['Datetime', '10Y_Yield']], on='Datetime', how='inner')

# 5. Calculate Spread (Yield - Inflation)
merged['spread_10Y_minus_inflation'] = merged['10Y_Yield'] - merged['inflation']

# Format Dates to standard YYYY-MM-DD
merged['period_date'] = merged['period_date'].dt.strftime('%Y-%m-%d')
merged['cpi_release_date'] = merged['cpi_release_date'].dt.strftime('%Y-%m-%d')

# Sort reverse-chronologically
merged.sort_values('Datetime', ascending=False, inplace=True)

# Select final columns and save
final_cols = ['period_date', 'cpi_release_date', 'inflation', '10Y_Yield', 'spread_10Y_minus_inflation']
final_df = merged[final_cols]

output_file = "CPI_GSec_Merged_Data.csv"
final_df.to_csv(output_file, index=False)

print(f"Data processed and saved to '{output_file}'!")

Data processed and saved to 'CPI_GSec_Merged_Data.csv'!


/var/folders/q5/pq31bcf10fv9ktv6b14g42zr0000gn/T/ipykernel_25516/932917289.py:23: PerformanceWarning: Non-vectorized DateOffset being applied to Series or DatetimeIndex.
  cpi_df['raw_release'] = cpi_df['Datetime'] + pd.DateOffset(months=1, day=14)


In [4]:
import pandas as pd

# 1. Load the Base Calendar
base_df = pd.read_csv('Monthly_Calendar_2010_2026.csv')
base_df['Last Day of Month'] = pd.to_datetime(base_df['Last Day of Month']).dt.normalize()
base_df.sort_values('Last Day of Month', inplace=True)

# 2. Load Yield Data
yield_df = pd.read_csv('Processed_G-Sec_Yield_With_Metrics.csv')
yield_df['release_date'] = pd.to_datetime(yield_df['release_date']).dt.normalize()
yield_df.sort_values('release_date', inplace=True)

# 3. Load CPI-GSec Data
cpi_gsec_df = pd.read_csv('CPI_GSec_Merged_Data.csv')
cpi_gsec_df['cpi_release_date'] = pd.to_datetime(cpi_gsec_df['cpi_release_date']).dt.normalize()
cpi_gsec_df.sort_values('cpi_release_date', inplace=True)

# 4. Load Repo Rate Data
repo_df = pd.read_csv('Repo_Rate_3M_Average.csv')
repo_df['Release_Date'] = pd.to_datetime(repo_df['Release_Date']).dt.normalize()
repo_df.sort_values('Release_Date', inplace=True)

# 5. Merge A: Attach Yield Metrics
merged_df = pd.merge_asof(
    base_df,
    yield_df[['release_date', '10Y_3Y_Slope', '10Y_3M_Chg', '10Y']],
    left_on='Last Day of Month',
    right_on='release_date',
    direction='backward'
)

# 6. Merge B: Attach CPI-GSec Spread
merged_df = pd.merge_asof(
    merged_df,
    cpi_gsec_df[['cpi_release_date', 'spread_10Y_minus_inflation']],
    left_on='Last Day of Month',
    right_on='cpi_release_date',
    direction='backward'
)

# 7. Merge C: Attach Repo Rate
merged_df = pd.merge_asof(
    merged_df,
    repo_df[['Release_Date', 'Repo_Rate_3M_Average', 'Repo_Rate']],
    left_on='Last Day of Month',
    right_on='Release_Date',
    direction='backward'
)

# 8. Clean up redundant date columns
merged_df.drop(columns=['release_date', 'cpi_release_date', 'Release_Date'], inplace=True)

# 9. Sort reverse chronological
merged_df.sort_values('Last Day of Month', ascending=False, inplace=True)

# 10. Save
merged_df.to_csv('Master_Macro_Calendar.csv', index=False)
print("Success! Master file created: Master_Macro_Calendar.csv")
print("\nQuick sanity check:")
print(merged_df.head())
print("\nNull counts:")
print(merged_df.isnull().sum())

Success! Master file created: Master_Macro_Calendar.csv

Quick sanity check:
    Last Day of Month  10Y_3Y_Slope  10Y_3M_Chg     10Y  \
196        2026-05-31        0.4277      0.3148  7.0267   
195        2026-04-30        0.6594      0.2999  7.0796   
194        2026-03-31        0.5939      0.4192  7.0782   
193        2026-02-28        0.7990      0.1091  6.7119   
192        2026-01-31        0.7535      0.1545  6.7797   

     spread_10Y_minus_inflation  Repo_Rate_3M_Average  Repo_Rate  
196                      3.5996              0.000000       5.25  
195                      3.6782             -0.083333       5.25  
194                      3.5019             -0.166667       5.25  
193                      4.0497             -0.250000       5.25  
192                      5.3290             -0.166667       5.25  

Null counts:
Last Day of Month              0
10Y_3Y_Slope                   0
10Y_3M_Chg                     0
10Y                            0
spread_10Y_minus_inf

In [5]:
import pandas as pd
import numpy as np

def calculate_macro_winsorized_zscores(file_path, output_path):
    # 1. Load data and parse dates
    df = pd.read_csv(file_path)
    df['Last Day of Month'] = pd.to_datetime(df['Last Day of Month'])
    df = df.sort_values('Last Day of Month', ascending=False).set_index('Last Day of Month')

    # Isolate only numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df_numeric = df[numeric_cols]

    # 3. Robust Z-score function
    def calc_robust_z_winsorized(window_slice):
        current_val = window_slice[-1]
        if np.isnan(current_val):
            return np.nan
        valid_vals = window_slice[~np.isnan(window_slice)]
        if len(valid_vals) == 0:
            return np.nan
        med = np.median(valid_vals)
        mad = np.median(np.abs(valid_vals - med))
        if mad == 0:
            z = 0.0  # entire window is identical, current value is perfectly average
        else:
            z = (current_val - med) / (1.4826 * mad)
        if not np.isnan(z):
            z = np.clip(z, -4.0, 4.0)
        return z

    # 4. Apply rolling calculation
    value_cols = ['Repo_Rate_3M_Average', '10Y_3Y_Slope', '10Y', 'spread_10Y_minus_inflation']
    z_df = pd.DataFrame(index=df_numeric.index)
    print("Columns in df_numeric:", df_numeric.columns.tolist())

    for col in value_cols:
        if col == 'Repo_Rate_3M_Average':
            # Scale by fixed constant instead of z-scoring — step function doesn't z-score well
            z_df[f'{col}_Score'] = (df_numeric[col] * 9).clip(-4, 4)
        elif col == '10Y_3Y_Slope':
            # Flip: steeper curve = easier, not tighter
            z_df[f'{col}_Z_Score'] = df_numeric[col].rolling(window=36, min_periods=18).apply(calc_robust_z_winsorized, raw=True)
        elif col == 'spread_10Y_minus_inflation':
            # Flip: steeper curve = easier, not tighter
            z_df[f'{col}_Z_Score'] = df_numeric[col].rolling(window=36, min_periods=18).apply(calc_robust_z_winsorized, raw=True)
        else:
            # Keep as-is: 10Y_3M_Chg = tighter
            z_df[f'{col}_Z_Score'] = df_numeric[col].rolling(window=36, min_periods=18).apply(calc_robust_z_winsorized, raw=True)

    # 5. Clean up
    z_scores = z_df.sort_index(ascending=False)
    z_scores.reset_index(inplace=True)

    # Save
    z_scores.to_csv(output_path, index=False)
    print(f"Success! Winsorized Z-Scores saved to: {output_path}")

# Run the function
calculate_macro_winsorized_zscores(
    file_path="Master_Macro_Calendar.csv",
    output_path="Master_Macro_Calendar_Winsorized_ZScores.csv"
)

Columns in df_numeric: ['10Y_3Y_Slope', '10Y_3M_Chg', '10Y', 'spread_10Y_minus_inflation', 'Repo_Rate_3M_Average', 'Repo_Rate']
Success! Winsorized Z-Scores saved to: Master_Macro_Calendar_Winsorized_ZScores.csv


In [6]:
import pandas as pd

def apply_tiered_color_formatting(input_csv, output_xlsx):
    # 1. Load the data
    df = pd.read_csv(input_csv)
    
    # 2. Isolate the Z-score columns to apply formatting (exclude the Date column)
    date_col = 'Last Day of Month'
    z_cols = [col for col in df.columns if col != date_col]
    
    # 3. Define the exact Tiered Formatting Logic
    def format_outliers_tiered(s):
        styles = []
        for val in s:
            if pd.isna(val):
                styles.append('')
                continue  
            try:
                val = float(val)
            except ValueError:
                styles.append('')
                continue
                
            # --- HIGH EXTREMES (RED) ---
            if val >= 2.0:
                # Severe Positive Outlier: Dark red fill, white text, bold
                styles.append('background-color: #990000; color: white; font-weight: bold;')
            elif 1.0 <= val < 2.0:
                # Mild Positive Outlier: Lighter soft red fill, darker red text
                styles.append('background-color: #ffe6e6; color: #990000;')
                
            # --- LOW EXTREMES (GREEN) ---
            elif val <= -2.0:
                # Severe Negative Outlier: Dark green fill, white text, bold
                styles.append('background-color: #006600; color: white; font-weight: bold;')
            elif -2.0 < val <= -1.0:
                # Mild Negative Outlier: Lighter soft green fill, darker green text
                styles.append('background-color: #e6ffe6; color: #006600;')
                
            # --- NORMAL RANGE ---
            else:
                styles.append('')
        return styles

    # 4. Apply the styling strictly to our data columns
    styled_df = df.style.apply(format_outliers_tiered, subset=z_cols, axis=0)
    
    # 5. Format cells to precisely 4 decimal places for cleanliness
    styled_df = styled_df.format({col: "{:.4f}" for col in z_cols})
    
    # 6. Export to Excel format
    styled_df.to_excel(output_xlsx, index=False, engine='openpyxl')
    print(f"Success! Tiered Excel report saved to: {output_xlsx}")

# Run the standalone script on your newly created Z-scores CSV
apply_tiered_color_formatting(
    input_csv='Master_Macro_Calendar_Winsorized_ZScores.csv', 
    output_xlsx='Master_Macro_Calendar_Scores_Highlighted.xlsx'
)

Success! Tiered Excel report saved to: Master_Macro_Calendar_Scores_Highlighted.xlsx


In [7]:
import pandas as pd
import numpy as np
import shutil
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# 1. Setup & Data Ingestion
INPUT_FILE = 'Master_Macro_Calendar_Scores_Highlighted.xlsx'
OUTPUT_FILE = 'Rates_Regime_Final.xlsx'

shutil.copy(INPUT_FILE, OUTPUT_FILE)

df = pd.read_excel(OUTPUT_FILE)
df['Last Day of Month'] = pd.to_datetime(df['Last Day of Month'])
df = df.sort_values('Last Day of Month').reset_index(drop=True)

repo_col = 'Repo_Rate_3M_Average_Score'
slope_col = '10Y_3Y_Slope_Z_Score'
yield_col = '10Y_Z_Score'

# 2. Standardized Scoring Functions
def score(val):
    if pd.isna(val): 
        return 0.0
    if val >= 2: 
        return 1.0     # Strongly Stressed / High Vulnerability
    elif val >= 1.3: 
        return 0.75     # Moderately Stressed
    elif val >= 0.7: 
        return 0.5     # Moderately Stressed
    elif val >= 0.25: 
        return 0.25     # Moderately Stressed
    elif val <= -2: 
        return -1.0    # Strongly Favorable / High Cushion
    elif val <= -1.3: 
        return -0.75    # Strongly Favorable / High Cushion
    elif val <= -0.7: 
        return -0.5    # Strongly Favorable / High Cushion
    elif val <= -0.25: 
        return -0.25    # Moderately Favorable
    else: 
        return 0.0     # Neutral Anchor 

def compute_net_dynamic(r, s, y):
    repo_s = score(r)
    slope_s = score(s)
    yield_s = score(y)

    w_repo = 0.50
    w_yield = 0.25
    w_slope = 0.25 if (repo_s != 0 and np.sign(slope_s) == np.sign(repo_s)) else 0.0

    total_weight = w_repo + w_yield + w_slope
    slope_contribution = slope_s if w_slope > 0 else 0.0

    return ((repo_s * w_repo) + (yield_s * w_yield) + (slope_contribution * w_slope)) / total_weight

def map_to_7_point_band(net):
    if pd.isna(net):
        return 0.0
    if net >= 0.8:
        return 3.0
    elif net >= 0.5:
        return 2.0
    elif net >= 0.15:
        return 1.0
    elif net > -0.15:
        return 0.0
    elif net >= -0.5:
        return -1.0
    elif net >= -0.8:
        return -2.0
    else:
        return -3.0

def apply_2_of_3_with_median_fallback(regimes):
    confirmed = []
    current_confirmed = 0.0

    for i in range(len(regimes)):
        if i < 2:
            current_confirmed = regimes[i] if pd.notna(regimes[i]) else 0.0
            confirmed.append(current_confirmed)
            continue

        window = [regimes[i], regimes[i-1], regimes[i-2]]
        window = [v for v in window if pd.notna(v)]

        if len(window) == 0:
            confirmed.append(current_confirmed)
            continue

        counts = pd.Series(window).value_counts()
        highest_frequency = counts.iloc[0]

        if highest_frequency >= 2:
            current_confirmed = counts.index[0]
        else:
            current_confirmed = float(np.median(window))

        confirmed.append(current_confirmed)

    return confirmed

# 3. Model Pipeline Execution
flash_regimes = []
repo_directional_scores = []
net_scores = []

for idx, row in df.iterrows():
    r = row[repo_col]
    s = row[slope_col]
    y = row[yield_col]

    net_score = compute_net_dynamic(r, s, y)
    net_scores.append(net_score)
    flash_regimes.append(map_to_7_point_band(net_score))
    repo_directional_scores.append(score(r))

smoothed_regimes = apply_2_of_3_with_median_fallback(flash_regimes)

final_gated_regimes = []
for i in range(len(smoothed_regimes)):
    conf = smoothed_regimes[i]
    repo_s = repo_directional_scores[i]

    if repo_s < 0:
        gated = min(conf, 0.0)
    elif repo_s > 0:
        gated = max(conf, 0.0)
    else:
        gated = conf

    final_gated_regimes.append(gated)

df['Final_Net_Score'] = net_scores
df['Rates_Regime_Score'] = final_gated_regimes

# 4. Presentation & Excel Polishing
df_desc = df.sort_values('Last Day of Month', ascending=False).reset_index(drop=True)

wb = openpyxl.load_workbook(OUTPUT_FILE)
ws = wb.active

header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
thin_border = Border(
    left=Side(style='thin', color='D9D9D9'),
    right=Side(style='thin', color='D9D9D9'),
    top=Side(style='thin', color='D9D9D9'),
    bottom=Side(style='thin', color='D9D9D9')
)

net_col_letter = 'F'
net_col_idx = 6
regime_col_letter = 'G'
regime_col_idx = 7

ws[f'{net_col_letter}1'] = 'Final_Net_Score'
ws[f'{net_col_letter}1'].font = header_font
ws[f'{net_col_letter}1'].fill = header_fill
ws[f'{net_col_letter}1'].alignment = Alignment(horizontal="center", vertical="center")
ws[f'{net_col_letter}1'].border = thin_border

ws[f'{regime_col_letter}1'] = 'Rates_Regime_Score'
ws[f'{regime_col_letter}1'].font = header_font
ws[f'{regime_col_letter}1'].fill = header_fill
ws[f'{regime_col_letter}1'].alignment = Alignment(horizontal="center", vertical="center")
ws[f'{regime_col_letter}1'].border = thin_border

for i, (net_val, reg_val) in enumerate(zip(df_desc['Final_Net_Score'], df_desc['Rates_Regime_Score'])):
    net_cell = ws.cell(row=i+2, column=net_col_idx, value=net_val)
    net_cell.alignment = Alignment(horizontal="right")
    net_cell.border = thin_border
    net_cell.number_format = '0.00'

    reg_cell = ws.cell(row=i+2, column=regime_col_idx, value=reg_val)
    reg_cell.alignment = Alignment(horizontal="right")
    reg_cell.border = thin_border
    reg_cell.number_format = '0.00'

ws.column_dimensions[net_col_letter].width = 18
ws.column_dimensions[regime_col_letter].width = 22

wb.save(OUTPUT_FILE)
print(f"Process complete! Output successfully saved to: {OUTPUT_FILE}")
print("\nFinal Gated Distribution:")
print(pd.Series(final_gated_regimes).value_counts().sort_index())

Process complete! Output successfully saved to: Rates_Regime_Final.xlsx

Final Gated Distribution:
-2.0    33
-1.0    70
 0.0    16
 1.0    29
 2.0    36
 3.0    13
Name: count, dtype: int64
